# OmicsDRP results summary
**Outputs** (`outputs/`, one integrated per-fold table per evaluation setting; nothing else is saved):

| File | Content |
|---|---|
| `mixed_folds.csv`, `unseen_cell_folds.csv`, `unseen_drug_folds.csv` | every nested-CV outer-test fold: all OmicsDRP conditions, omics-free naive baselines, competitor models |
| `external_folds.csv` | DepMap (CCLE omics) x PRISM `mts`/`hts`: every fold model + the 5-fold `ensemble` row |

Section 2 computes every comparison from these four files and only displays it.

In [13]:
import json, glob, os, sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

def find_results_dir(start=None):
    start = Path(start or os.getcwd()).resolve()
    for base in [start, *start.parents]:
        for rel in ['scripts/Results', 'omicsdrp/scripts/Results', 'Results']:
            p = base / rel
            if p.is_dir():
                return p
    raise FileNotFoundError('Results dir not found from ' + str(start))

RESULTS_DIR = find_results_dir()
PKG_ROOT = RESULTS_DIR.parents[1]                     # omicsdrp/
BENCH_DIR = PKG_ROOT / 'benchmark' / 'BenchmarkResults'
EXTERNAL_DIR = BENCH_DIR / 'ccle_external'
INFERENCE_DIR = PKG_ROOT / 'scripts' / 'InferenceModels'
OUT = Path('outputs')
sys.path.insert(0, str(PKG_ROOT / 'src'))
from omicsdrp.metrics import regression_metrics

SPLITS = ['mixed', 'unseen_cell', 'unseen_drug']
METRICS = ['rmse', 'mae', 'r2', 'pearson', 'spearman']
OMICS_ORDER = ['SNP', 'MET', 'CNV', 'RNA']
CFG_COLS = ['omics', 'noise', 'genes', 'cell', 'drug']
print('RESULTS_DIR =', RESULTS_DIR)

RESULTS_DIR = /project/OmicsDRP_Review/omicsdrp/scripts/Results


## 1. Build the integrated per-fold outputs

In [14]:
def cfg_fields(cfg):
    # noise substitution + gene-set control are separate ablation axes: the omics
    # columns stay in place, so `omics` alone can't tell them apart.
    return {'omics': '+'.join(o for o in OMICS_ORDER if o in cfg['omics']),
            'noise': '+'.join(o for o in OMICS_ORDER if o in (cfg.get('noise_omics') or [])) or 'none',
            'genes': cfg.get('gene_set', 'pgkb'),
            'cell': cfg['cell_encoder'], 'drug': cfg['drug_encoder']}

def n_params(tag):
    p = RESULTS_DIR / tag / 'events.jsonl'
    if p.exists():
        for line in open(p):
            d = json.loads(line)
            if d.get('kind') == 'model_info':
                return d['n_params']
    return None

def omicsdrp_folds():
    rows = []
    for cfg_path in sorted(RESULTS_DIR.glob('*/config.json')):
        d = cfg_path.parent
        fold_jsons = sorted((d / 'folds').glob('fold_*_test_metrics.json'))
        if not fold_jsons:
            continue  # no completed folds yet
        cfg = json.load(open(cfg_path))
        base = {'split_mode': cfg['split_mode'], 'model': 'omicsdrp', **cfg_fields(cfg),
                'tag': d.name, 'n_params': n_params(d.name)}
        for fj in fold_jsons:
            r = json.load(open(fj))
            rows.append({**base, 'fold': int(r['fold']), 'best_epoch': r.get('best_epoch'),
                         **{m: r.get(m, np.nan) for m in METRICS}})
    return pd.DataFrame(rows)

def naive_folds(split_modes):
    # Omics-free predictors on the SAME outer-test folds: how much of the score
    # is bookkeeping (per-drug mean, plus a per-cell offset) rather than modelling.
    from omicsdrp.config import ExperimentConfig
    from omicsdrp.data import load_raw
    from omicsdrp.splits import build_folds
    data_path = str(PKG_ROOT.parent / 'data')
    raw = load_raw(data_path)
    ic = raw.ic50.numpy()
    rows = []
    for sm in split_modes:
        for f in build_folds(raw, ExperimentConfig(dataset_path=data_path, split_mode=sm)):
            tr = raw.pairs[np.concatenate([f.train_pair_idx, f.val_pair_idx])]
            te = raw.pairs[f.test_pair_idx]
            y_tr, y_te = ic[tr[:, 0], tr[:, 1]], ic[te[:, 0], te[:, 1]]
            gm = y_tr.mean()
            dm = np.full(raw.n_drug, gm)
            for d in np.unique(tr[:, 1]):
                dm[d] = y_tr[tr[:, 1] == d].mean()
            cdev = np.zeros(raw.n_cell)
            res = y_tr - dm[tr[:, 1]]
            for c in np.unique(tr[:, 0]):
                cdev[c] = res[tr[:, 0] == c].mean()
            for name, pred in (('naive_global_mean', np.full(len(y_te), gm)),
                               ('naive_drug_mean', dm[te[:, 1]]),
                               ('naive_drug_mean_cell_offset', dm[te[:, 1]] + cdev[te[:, 0]])):
                m = regression_metrics(y_te, pred)
                rows.append({'split_mode': sm, 'model': name, 'fold': f.fold,
                             **{k: m[k] for k in METRICS}})
    return pd.DataFrame(rows)

def competitor_folds():
    rows = []
    for split_dir in sorted(BENCH_DIR.glob('*/nested__*')):
        for p in sorted(split_dir.glob('fold_*/metrics.json')):
            r = json.load(open(p))
            rows.append({'split_mode': split_dir.name.replace('nested__', ''),
                         'model': split_dir.parent.name, 'fold': int(p.parent.name.split('_')[1]),
                         'n_params': r.get('n_params_total'), **{m: r.get(m, np.nan) for m in METRICS}})
    return pd.DataFrame(rows)

CV_COLS = ['model', *CFG_COLS, 'tag', 'fold', 'best_epoch', 'n_params', *METRICS]
od = omicsdrp_folds()
cv_all = pd.concat([od, naive_folds(SPLITS), competitor_folds()], ignore_index=True)
cv_all['n_params'] = cv_all['n_params'].astype('Int64')
cv_all['best_epoch'] = cv_all['best_epoch'].astype('Int64')
OUT.mkdir(exist_ok=True)
for sm in SPLITS:
    sub = (cv_all[cv_all.split_mode == sm][CV_COLS]
           .sort_values(['model', *CFG_COLS, 'fold'], na_position='first'))
    sub.to_csv(OUT / f'{sm}_folds.csv', index=False)
    print(f'saved {OUT / f"{sm}_folds.csv"}: {len(sub)} fold rows, '
          f'{sub.groupby(["model", "tag"], dropna=False).ngroups} experiments')

/project/OmicsDRP_Review/omicsdrp/src/omicsdrp/splits.py:167: UserWarning: [unseen_drug] 4/8 clusters have < 5 members; those groups cannot be stratified across 5 folds and will land in a single test fold. Consider lowering n_cluster_drug (inspect via scripts/inspect_clusters.py).
  warnings.warn(
/home/mjcho/miniconda3/envs/omicsdrp/lib/python3.9/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/home/mjcho/miniconda3/envs/omicsdrp/lib/python3.9/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=7.
  warnings.warn(
/home/mjcho/miniconda3/envs/omicsdrp/lib/python3.9/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=7.
  warnings.warn(
/home/mjcho/miniconda3/envs/omicsdrp/lib/python3.9/si

saved outputs/mixed_folds.csv: 210 fold rows, 42 experiments
saved outputs/unseen_cell_folds.csv: 285 fold rows, 57 experiments
saved outputs/unseen_drug_folds.csv: 210 fold rows, 42 experiments


In [15]:
def parse_external(stem):
    # '<model>__<split>' (competitor) or 'omicsdrp__<tag>__<split>'
    parts = stem.split('__')
    return parts[0], ('__'.join(parts[1:-1]) or None), parts[-1]

def external_folds():
    rows = []
    for p in sorted(EXTERNAL_DIR.glob('*_predictions.parquet')):
        model, tag, dataset = parse_external(p.name[:-len('_predictions.parquet')])
        cfg = cfg_fields(json.load(open(INFERENCE_DIR / tag / 'config.json'))) if tag else {}
        df = pd.read_parquet(p)
        preds = {c[len('pred_fold'):]: df[c].values for c in df.columns if c.startswith('pred_fold')}
        preds['ensemble'] = df['pred'].values
        for fold, pred in preds.items():
            m = regression_metrics(df['true'].values, pred)
            rows.append({'dataset': dataset, 'model': model, **cfg, 'tag': tag, 'fold': fold,
                         'n_pairs': len(df), **{k: m[k] for k in METRICS}})
    return pd.DataFrame(rows)

ext_all = external_folds()
ext_all = ext_all[['dataset', 'model', *CFG_COLS, 'tag', 'fold', 'n_pairs', *METRICS]]
ext_all = ext_all.sort_values(['dataset', 'model', 'tag', 'fold'], na_position='first')
ext_all.to_csv(OUT / 'external_folds.csv', index=False)
print(f'saved {OUT / "external_folds.csv"}: {len(ext_all)} rows '
      f'({ext_all.groupby(["dataset", "model", "tag"], dropna=False).ngroups} model x dataset)')

saved outputs/external_folds.csv: 216 rows (36 model x dataset)


## 2. Comparisons (computed from `outputs/*_folds.csv`, display only)
Δ columns are **paired per outer fold** (same folds across conditions), then averaged.

In [16]:
cv = pd.concat([pd.read_csv(OUT / f'{sm}_folds.csv').assign(split_mode=sm) for sm in SPLITS],
               ignore_index=True).astype({'n_params': 'Int64', 'best_epoch': 'Int64'})
ext = pd.read_csv(OUT / 'external_folds.csv')
omics_cv = cv[cv.model == 'omicsdrp']

def summarize(df, keys, metrics=METRICS):
    g = df.groupby(keys, dropna=False, observed=True)
    s = g[metrics].agg(['mean', 'std'])
    s.columns = [f'{m}_{a}' for m, a in s.columns]
    return s.join(g['fold'].nunique().rename('n_folds')).reset_index()

def pm(s, metrics=METRICS):
    # 'mean ± std' display table
    out = s.drop(columns=[c for c in s.columns if c.endswith(('_mean', '_std'))]).copy()
    for m in metrics:
        out[m] = s[f'{m}_mean'].map('{:.4f}'.format) + ' ± ' + s[f'{m}_std'].map('{:.4f}'.format)
    return out

def paired_delta(df, ref_mask, on):
    # adds d_<metric> = row − reference row on the same `on` keys (incl. fold)
    ref = df[ref_mask].set_index(on)[METRICS]
    out = df.join(ref, on=on, rsuffix='_ref')
    for m in METRICS:
        out[f'd_{m}'] = out[m] - out.pop(f'{m}_ref')
    return out

BASE = dict(omics='SNP+MET+CNV+RNA', noise='none', genes='pgkb', cell='attention', drug='morgan')
print(cv.groupby(['split_mode', 'model']).size().unstack(0).fillna(0).astype(int))

split_mode                   mixed  unseen_cell  unseen_drug
model                                                       
deeptta                          5            5            5
graphdrp                         5            5            5
graphdrp_gat                     5            5            5
graphdrp_gat_gcn                 5            5            5
graphdrp_gcn                     5            5            5
naive_drug_mean                  5            5            5
naive_drug_mean_cell_offset      5            5            5
naive_global_mean                5            5            5
omicsdrp                       165          240          165
paccmann                         5            5            5


### 2.1 All OmicsDRP conditions (mean ± std over folds)

In [17]:
for sm in SPLITS:
    s = summarize(omics_cv[omics_cv.split_mode == sm], CFG_COLS).sort_values('rmse_mean')
    print(f'\n--- {sm}: {len(s)} conditions ---')
    display(pm(s))


--- mixed: 33 conditions ---


,omics,noise,genes,cell,drug,n_folds,rmse,mae,r2,pearson,spearman
27,SNP+MET+CNV+RNA,none,random:2,attention,morgan,5,0.9633 ± 0.0068,0.7132 ± 0.0057,0.8839 ± 0.0024,0.9403 ± 0.0013,0.9141 ± 0.0015
22,SNP+MET+CNV+RNA,none,pgkb,attention,morgan,5,0.9652 ± 0.0235,0.7156 ± 0.0177,0.8834 ± 0.0065,0.9401 ± 0.0035,0.9143 ± 0.0038
7,SNP+MET+CNV+RNA,MET+CNV,pgkb,attention,morgan,5,0.9657 ± 0.0063,0.7142 ± 0.0027,0.8833 ± 0.0022,0.9399 ± 0.0011,0.9142 ± 0.0012
0,CNV+RNA,none,pgkb,attention,morgan,5,0.9662 ± 0.0116,0.7160 ± 0.0088,0.8832 ± 0.0038,0.9399 ± 0.0020,0.9139 ± 0.0019
4,SNP+MET+CNV+RNA,CNV,pgkb,attention,morgan,5,0.9669 ± 0.0149,0.7137 ± 0.0106,0.8830 ± 0.0045,0.9399 ± 0.0023,0.9139 ± 0.0024
23,SNP+MET+CNV+RNA,none,pgkb,attention,unimol,5,0.9671 ± 0.0118,0.7187 ± 0.0099,0.8830 ± 0.0033,0.9399 ± 0.0019,0.9139 ± 0.0025
13,SNP+MET+CNV+RNA,SNP+CNV+RNA,pgkb,attention,morgan,5,0.9672 ± 0.0071,0.7151 ± 0.0037,0.8829 ± 0.0028,0.9398 ± 0.0015,0.9138 ± 0.0014
15,SNP+MET+CNV+RNA,SNP+MET+CNV,pgkb,attention,morgan,5,0.9674 ± 0.0122,0.7135 ± 0.0076,0.8829 ± 0.0039,0.9398 ± 0.0020,0.9140 ± 0.0017
21,SNP+MET+CNV+RNA,none,pgkb,attention,molformer,5,0.9688 ± 0.0083,0.7171 ± 0.0060,0.8826 ± 0.0029,0.9398 ± 0.0015,0.9139 ± 0.0017
5,SNP+MET+CNV+RNA,CNV+RNA,pgkb,attention,morgan,5,0.9691 ± 0.0130,0.7180 ± 0.0090,0.8825 ± 0.0040,0.9396 ± 0.0020,0.9138 ± 0.0018



--- unseen_cell: 48 conditions ---


,omics,noise,genes,cell,drug,n_folds,rmse,mae,r2,pearson,spearman
45,SNP+MET+CNV+RNA,none,random:5,attention,morgan,5,1.2896 ± 0.0229,0.9691 ± 0.0176,0.7919 ± 0.0046,0.8901 ± 0.0025,0.8462 ± 0.0037
43,SNP+MET+CNV+RNA,none,random:3,attention,morgan,5,1.3046 ± 0.0304,0.9766 ± 0.0208,0.7869 ± 0.0084,0.8878 ± 0.0047,0.8438 ± 0.0058
38,SNP+MET+CNV+RNA,none,pgkb,attention,unimol,5,1.3050 ± 0.0163,0.9776 ± 0.0112,0.7869 ± 0.0030,0.8879 ± 0.0019,0.8447 ± 0.0028
42,SNP+MET+CNV+RNA,none,random:2,attention,morgan,5,1.3058 ± 0.0271,0.9751 ± 0.0199,0.7866 ± 0.0082,0.8882 ± 0.0041,0.8448 ± 0.0046
41,SNP+MET+CNV+RNA,none,random:1,attention,morgan,5,1.3075 ± 0.0322,0.9794 ± 0.0266,0.7859 ± 0.0099,0.8875 ± 0.0048,0.8436 ± 0.0060
40,SNP+MET+CNV+RNA,none,pgkb,mlp,morgan,5,1.3080 ± 0.0353,0.9804 ± 0.0238,0.7859 ± 0.0082,0.8871 ± 0.0045,0.8437 ± 0.0048
19,SNP+MET+CNV+RNA,SNP,pgkb,attention,morgan,5,1.3083 ± 0.0269,0.9808 ± 0.0205,0.7858 ± 0.0055,0.8871 ± 0.0031,0.8434 ± 0.0043
37,SNP+MET+CNV+RNA,none,pgkb,attention,morgan,5,1.3086 ± 0.0314,0.9813 ± 0.0236,0.7857 ± 0.0072,0.8871 ± 0.0039,0.8430 ± 0.0042
36,SNP+MET+CNV+RNA,none,pgkb,attention,molformer,5,1.3090 ± 0.0301,0.9815 ± 0.0217,0.7856 ± 0.0072,0.8868 ± 0.0039,0.8431 ± 0.0043
44,SNP+MET+CNV+RNA,none,random:4,attention,morgan,5,1.3093 ± 0.0176,0.9800 ± 0.0123,0.7854 ± 0.0075,0.8869 ± 0.0045,0.8433 ± 0.0073



--- unseen_drug: 33 conditions ---


,omics,noise,genes,cell,drug,n_folds,rmse,mae,r2,pearson,spearman
19,SNP+MET+CNV+RNA,none,pgkb,attention,chemberta,5,2.5355 ± 0.1410,1.9110 ± 0.0704,0.1797 ± 0.0798,0.5051 ± 0.0722,0.4781 ± 0.0713
21,SNP+MET+CNV+RNA,none,pgkb,attention,molformer,5,2.5363 ± 0.1972,1.9200 ± 0.1498,0.1799 ± 0.0903,0.4954 ± 0.0653,0.4373 ± 0.0495
24,SNP+MET+CNV+RNA,none,pgkb,mlp,molformer,5,2.5669 ± 0.1811,1.9478 ± 0.0887,0.1568 ± 0.1203,0.4875 ± 0.0970,0.4351 ± 0.0597
23,SNP+MET+CNV+RNA,none,pgkb,attention,unimol,5,2.5723 ± 0.2508,1.9595 ± 0.2060,0.1594 ± 0.0863,0.4473 ± 0.0650,0.4379 ± 0.0728
13,SNP+MET+CNV+RNA,SNP+CNV+RNA,pgkb,attention,morgan,5,2.6055 ± 0.2594,1.9475 ± 0.1535,0.1342 ± 0.1245,0.4478 ± 0.1183,0.3978 ± 0.0657
11,SNP+MET+CNV+RNA,SNP,pgkb,attention,morgan,5,2.6066 ± 0.2948,1.9465 ± 0.1773,0.1328 ± 0.1439,0.4342 ± 0.1210,0.3970 ± 0.0953
29,SNP+MET+CNV+RNA,none,random:4,attention,morgan,5,2.6120 ± 0.2663,1.9584 ± 0.1560,0.1308 ± 0.1195,0.4428 ± 0.1004,0.3934 ± 0.0782
25,SNP+MET+CNV+RNA,none,pgkb,mlp,morgan,5,2.6157 ± 0.2472,1.9787 ± 0.1599,0.1275 ± 0.1171,0.4449 ± 0.1083,0.3996 ± 0.0839
10,SNP+MET+CNV+RNA,RNA,pgkb,attention,morgan,5,2.6175 ± 0.2735,1.9516 ± 0.1590,0.1261 ± 0.1306,0.4327 ± 0.1111,0.3952 ± 0.0854
6,SNP+MET+CNV+RNA,MET,pgkb,attention,morgan,5,2.6250 ± 0.2807,1.9715 ± 0.1732,0.1215 ± 0.1308,0.4342 ± 0.1195,0.3848 ± 0.1039


### 2.2 One-factor-at-a-time ablations around the baseline, with naive baselines
Δ = condition − baseline (morgan, attention, all omics, PGKB genes) on the same regime; `Δ/σ` divides by the condition's own fold sd.
Feature noise (v2: N(0,1) substitution, same parameters) and feature drop (v1: column removed, fewer parameters) stay separate families.

In [18]:
def condition(r):
    if r.model != 'omicsdrp':
        return r.model, 'naive'
    for col, label, fam in (('genes', 'genes={}', 'gene_set'),
                            ('noise', 'noise={}', 'feature_noise_v2'),
                            ('omics', 'drop->{}', 'feature_drop_v1'),
                            ('cell', 'cell={}', 'cell_encoder'),
                            ('drug', 'drug={}', 'drug_encoder')):
        if r[col] != BASE[col]:
            return label.format(r[col]), fam
    return 'baseline', 'baseline'

# OFAT only: a row differing from BASE on >1 axis (e.g. noise=X with molformer)
# would share a label with the morgan row; those are in 2.3 / 2.4.
ofat = omics_cv[(omics_cv[CFG_COLS] != pd.Series(BASE)).sum(axis=1) <= 1]
naive = cv[cv.model.str.startswith('naive')]
u = pd.concat([ofat, naive])
u[['condition', 'family']] = u.apply(condition, axis=1, result_type='expand')
unified = summarize(u, ['split_mode', 'family', 'condition'])
base_rmse = unified[unified.condition == 'baseline'].set_index('split_mode')['rmse_mean']
unified['rmse_delta'] = unified.rmse_mean - unified.split_mode.map(base_rmse)
unified['rmse_delta_sigma'] = unified.rmse_delta / unified.rmse_std.replace(0, np.nan)
display(unified.sort_values(['split_mode', 'rmse_mean'])[
    ['split_mode', 'family', 'condition', 'rmse_mean', 'rmse_std', 'r2_mean', 'pearson_mean',
     'rmse_delta', 'rmse_delta_sigma', 'n_folds']].round(4))

FAMILY_ORDER = ['baseline', 'naive', 'feature_noise_v2', 'feature_drop_v1',
                'gene_set', 'cell_encoder', 'drug_encoder']
wide = unified.pivot_table(index=['family', 'condition'], columns='split_mode',
                           values=['rmse_mean', 'rmse_std', 'rmse_delta_sigma'])
wide = wide.swaplevel(axis=1).sort_index(axis=1, level=0)
wide = wide.reindex([f for f in FAMILY_ORDER if f in wide.index.get_level_values(0)], level=0)
print('\nwide view (paper table):')
display(wide.round(4))

,split_mode,family,condition,rmse_mean,rmse_std,r2_mean,pearson_mean,rmse_delta,rmse_delta_sigma,n_folds
28,mixed,gene_set,genes=random:2,0.9633,0.0068,0.8839,0.9403,-0.0019,-0.2799,5
0,mixed,baseline,baseline,0.9652,0.0235,0.8834,0.9401,0.0000,0.0000,5
15,mixed,feature_noise_v2,noise=MET+CNV,0.9657,0.0063,0.8833,0.9399,0.0005,0.0786,5
6,mixed,feature_drop_v1,drop->CNV+RNA,0.9662,0.0116,0.8832,0.9399,0.0010,0.0841,5
12,mixed,feature_noise_v2,noise=CNV,0.9669,0.0149,0.8830,0.9399,0.0018,0.1180,5
...,...,...,...,...,...,...,...,...,...,...
103,unseen_drug,naive,naive_drug_mean_cell_offset,2.6948,0.1370,0.0755,0.3148,0.0418,0.3054,5
73,unseen_drug,drug_encoder,drug=graphormer,2.6965,0.1590,0.0693,0.4326,0.0435,0.2733,5
83,unseen_drug,feature_noise_v2,noise=CNV+RNA,2.7048,0.2352,0.0666,0.3993,0.0518,0.2202,5
102,unseen_drug,naive,naive_drug_mean,2.8355,0.1332,-0.0235,0.0000,0.1824,1.3699,5



wide view (paper table):


split_mode                                              mixed                         unseen_cell                         unseen_drug                   
                                             rmse_delta_sigma rmse_mean rmse_std rmse_delta_sigma rmse_mean rmse_std rmse_delta_sigma rmse_mean rmse_std
family           condition                                                                                                                              
baseline         baseline                              0.0000    0.9652   0.0235           0.0000    1.3086   0.0314           0.0000    2.6530   0.2463
naive            naive_drug_mean                     141.6541    1.4959   0.0037           3.6262    1.4943   0.0512           1.3699    2.8355   0.1332
                 naive_drug_mean_cell_offset          38.1863    1.2074   0.0063           3.6262    1.4943   0.0512           0.3054    2.6948   0.1370
                 naive_global_mean                   141.4709    2.8272   0.0132          66.1252    2.8272   0.0230           1.3699    2.8355   0.1332
feature_noise_v2 noise=CNV                             0.1180    0.9669   0.0149           0.2145    1.3177   0.0422           0.0807    2.6726   0.2421
                 noise=CNV+RNA                         0.3045    0.9691   0.0130           1.1031    1.3624   0.0488           0.2202    2.7048   0.2352
                 noise=MET                             1.1694    0.9749   0.0083           0.3246    1.3237   0.0466          -0.0999    2.6250   0.2807
                 noise=MET+CNV                         0.0786    0.9657   0.0063           0.3757    1.3222   0.0362           0.0273    2.6605   0.2752
                 noise=MET+CNV+RNA                     1.5548    0.9845   0.0124           3.9183    1.5471   0.0609           0.0656    2.6684   0.2345
                 noise=MET+RNA                         1.2269    0.9833   0.0148           4.0315    1.4843   0.0436           0.0180    2.6570   0.2187
                 noise=RNA                             1.2433    0.9758   0.0086           2.1105    1.3560   0.0225          -0.1297    2.6175   0.2735
                 noise=SNP                             2.4432    0.9806   0.0063          -0.0103    1.3083   0.0269          -0.1575    2.6066   0.2948
                 noise=SNP+CNV                         0.3999    0.9715   0.0157           0.0736    1.3103   0.0231          -0.0180    2.6488   0.2353
                 noise=SNP+CNV+RNA                     0.2795    0.9672   0.0071           1.6479    1.3616   0.0321          -0.1833    2.6055   0.2594
                 noise=SNP+MET                         0.7871    0.9755   0.0131           0.0778    1.3098   0.0148           0.0634    2.6682   0.2398
                 noise=SNP+MET+CNV                     0.1789    0.9674   0.0122           0.5596    1.3268   0.0325          -0.0314    2.6447   0.2633
                 noise=SNP+MET+CNV+RNA                 1.7322    0.9892   0.0139           4.6189    1.5525   0.0528          -0.0945    2.6281   0.2635
                 noise=SNP+MET+RNA                     1.2611    0.9851   0.0158           3.2895    1.4759   0.0509           0.1421    2.6878   0.2447
                 noise=SNP+RNA                         0.5768    0.9740   0.0152           1.4902    1.3476   0.0262           0.0366    2.6617   0.2367
feature_drop_v1  drop->CNV+RNA                         0.0841    0.9662   0.0116           0.9127    1.3369   0.0310           0.1625    2.6913   0.2355
                 drop->MET+CNV+RNA                     0.6130    0.9747   0.0155           0.8280    1.3265   0.0216           0.0988    2.6802   0.2756
                 drop->MET+RNA                         0.3933    0.9700   0.0122           0.9456    1.3179   0.0098          -0.0152    2.6478   0.3405
                 drop->SNP+CNV+RNA                     0.6036    0.9739   0.0144           1.8786    1.3260   0.0092           0.0914    2.6734   0.2229
                 drop-

### 2.3 MolFormer + omics noise ablation (unseen_cell)
Produced by `./run_sweep.sh --split_mode unseen_cell --drug_encoder molformer --stages 0_baseline 1_feature`.
Δ is paired vs the same drug encoder's `noise=none` run.

In [19]:
MF_SPLIT, MF_DRUGS = 'unseen_cell', ['morgan', 'molformer']
NOISE_ORDER = ['none', 'SNP', 'MET', 'CNV', 'RNA', 'SNP+MET', 'SNP+CNV', 'SNP+RNA',
               'MET+CNV', 'MET+RNA', 'CNV+RNA', 'SNP+MET+CNV', 'SNP+MET+RNA',
               'SNP+CNV+RNA', 'MET+CNV+RNA', 'SNP+MET+CNV+RNA']

mf = omics_cv[(omics_cv.split_mode == MF_SPLIT) & omics_cv.drug.isin(MF_DRUGS)
              & (omics_cv.omics == BASE['omics']) & (omics_cv.genes == 'pgkb')
              & (omics_cv.cell == 'attention')].copy()
mf['noise'] = pd.Categorical(mf['noise'], NOISE_ORDER, ordered=True)
mf = paired_delta(mf, mf.noise == 'none', ['drug', 'fold'])
mf_s = summarize(mf, ['drug', 'noise'], METRICS + [f'd_{m}' for m in METRICS])

for drug in MF_DRUGS:
    t = mf_s[mf_s.drug == drug].set_index('noise')
    full = pm(t[[c for c in t.columns if not c.startswith('d_')]])
    for m in METRICS:
        full[f'Δ{m}'] = t[f'd_{m}_mean'].map('{:+.4f}'.format)
    print(f'\n--- {MF_SPLIT} | drug={drug} (Δ vs own noise=none) ---')
    display(full.drop(columns='drug'))


--- unseen_cell | drug=morgan (Δ vs own noise=none) ---


,n_folds,rmse,mae,r2,pearson,spearman,Δrmse,Δmae,Δr2,Δpearson,Δspearman
noise,,,,,,,,,,,
none,5,1.3086 ± 0.0314,0.9813 ± 0.0236,0.7857 ± 0.0072,0.8871 ± 0.0039,0.8430 ± 0.0042,+0.0000,+0.0000,+0.0000,+0.0000,+0.0000
SNP,5,1.3083 ± 0.0269,0.9808 ± 0.0205,0.7858 ± 0.0055,0.8871 ± 0.0031,0.8434 ± 0.0043,-0.0003,-0.0005,+0.0001,+0.0000,+0.0003
MET,5,1.3237 ± 0.0466,0.9917 ± 0.0345,0.7807 ± 0.0127,0.8844 ± 0.0066,0.8398 ± 0.0077,+0.0151,+0.0104,-0.0051,-0.0027,-0.0032
CNV,5,1.3177 ± 0.0422,0.9904 ± 0.0354,0.7827 ± 0.0113,0.8854 ± 0.0059,0.8412 ± 0.0075,+0.0091,+0.0091,-0.0030,-0.0017,-0.0018
RNA,5,1.3560 ± 0.0225,1.0151 ± 0.0203,0.7699 ± 0.0053,0.8790 ± 0.0038,0.8339 ± 0.0051,+0.0474,+0.0338,-0.0158,-0.0080,-0.0092
SNP+MET,5,1.3098 ± 0.0148,0.9814 ± 0.0117,0.7854 ± 0.0016,0.8866 ± 0.0009,0.8427 ± 0.0013,+0.0011,+0.0001,-0.0004,-0.0005,-0.0003
SNP+CNV,5,1.3103 ± 0.0231,0.9808 ± 0.0186,0.7852 ± 0.0048,0.8864 ± 0.0028,0.8421 ± 0.0047,+0.0017,-0.0005,-0.0005,-0.0006,-0.0010
SNP+RNA,5,1.3476 ± 0.0262,1.0094 ± 0.0178,0.7728 ± 0.0056,0.8798 ± 0.0037,0.8343 ± 0.0058,+0.0390,+0.0281,-0.0130,-0.0073,-0.0088
MET+CNV,5,1.3222 ± 0.0362,0.9908 ± 0.0244,0.7812 ± 0.0088,0.8842 ± 0.0049,0.8396 ± 0.0055,+0.0136,+0.0096,-0.0045,-0.0029,-0.0035



--- unseen_cell | drug=molformer (Δ vs own noise=none) ---


,n_folds,rmse,mae,r2,pearson,spearman,Δrmse,Δmae,Δr2,Δpearson,Δspearman
noise,,,,,,,,,,,
none,5,1.3090 ± 0.0301,0.9815 ± 0.0217,0.7856 ± 0.0072,0.8868 ± 0.0039,0.8431 ± 0.0043,+0.0000,+0.0000,+0.0000,+0.0000,+0.0000
SNP,5,1.3223 ± 0.0341,0.9950 ± 0.0277,0.7812 ± 0.0095,0.8851 ± 0.0040,0.8405 ± 0.0046,+0.0134,+0.0135,-0.0044,-0.0017,-0.0026
MET,5,1.3293 ± 0.0294,0.9970 ± 0.0240,0.7789 ± 0.0067,0.8834 ± 0.0042,0.8382 ± 0.0053,+0.0203,+0.0154,-0.0067,-0.0034,-0.0049
CNV,5,1.3163 ± 0.0321,0.9897 ± 0.0289,0.7832 ± 0.0077,0.8854 ± 0.0043,0.8407 ± 0.0056,+0.0073,+0.0081,-0.0024,-0.0013,-0.0024
RNA,5,1.3762 ± 0.0149,1.0321 ± 0.0135,0.7630 ± 0.0045,0.8748 ± 0.0024,0.8275 ± 0.0042,+0.0673,+0.0506,-0.0226,-0.0120,-0.0156
SNP+MET,5,1.3236 ± 0.0254,0.9917 ± 0.0185,0.7808 ± 0.0054,0.8841 ± 0.0029,0.8393 ± 0.0045,+0.0147,+0.0102,-0.0048,-0.0026,-0.0038
SNP+CNV,5,1.3214 ± 0.0212,0.9911 ± 0.0131,0.7815 ± 0.0037,0.8848 ± 0.0022,0.8396 ± 0.0031,+0.0125,+0.0096,-0.0041,-0.0020,-0.0035
SNP+RNA,5,1.3564 ± 0.0314,1.0165 ± 0.0214,0.7698 ± 0.0075,0.8781 ± 0.0040,0.8316 ± 0.0053,+0.0474,+0.0350,-0.0158,-0.0086,-0.0115
MET+CNV,5,1.3285 ± 0.0247,0.9928 ± 0.0191,0.7792 ± 0.0051,0.8840 ± 0.0034,0.8398 ± 0.0041,+0.0195,+0.0112,-0.0064,-0.0028,-0.0033


### 2.4 Cell encoder: attention vs MLP (MolFormer and Morgan, all splits)
Produced by `python runner.py --split_mode <split> --drug_encoder molformer --stages 0_baseline 2_attention`.
Δ = MLP − attention, paired per fold.

In [20]:
CE_DRUGS = ['molformer', 'morgan']
ce = omics_cv[omics_cv.drug.isin(CE_DRUGS) & (omics_cv.omics == BASE['omics'])
              & (omics_cv.noise == 'none') & (omics_cv.genes == 'pgkb')].copy()
ce = paired_delta(ce, ce.cell == 'attention', ['split_mode', 'drug', 'fold'])
ce_s = summarize(ce, ['split_mode', 'drug', 'cell'], METRICS + [f'd_{m}' for m in METRICS])
ce_s['split_mode'] = pd.Categorical(ce_s.split_mode, SPLITS, ordered=True)
ce_s['drug'] = pd.Categorical(ce_s.drug, CE_DRUGS, ordered=True)
t = ce_s.sort_values(['split_mode', 'drug', 'cell']).set_index(['split_mode', 'drug', 'cell'])
full = pm(t[[c for c in t.columns if not c.startswith('d_')]])
for m in METRICS:
    full[f'Δ{m}'] = t[f'd_{m}_mean'].map('{:+.4f}'.format).where(
        t.index.get_level_values('cell') == 'mlp', '')
display(full)

n_folds             rmse              mae               r2          pearson         spearman    Δrmse     Δmae      Δr2 Δpearson Δspearman
split_mode  drug      cell                                                                                                                                                 
mixed       molformer attention        5  0.9688 ± 0.0083  0.7171 ± 0.0060  0.8826 ± 0.0029  0.9398 ± 0.0015  0.9139 ± 0.0017                                              
                      mlp              5  1.0143 ± 0.0219  0.7539 ± 0.0163  0.8712 ± 0.0061  0.9334 ± 0.0032  0.9046 ± 0.0033  +0.0456  +0.0367  -0.0114  -0.0063   -0.0092
            morgan    attention        5  0.9652 ± 0.0235  0.7156 ± 0.0177  0.8834 ± 0.0065  0.9401 ± 0.0035  0.9143 ± 0.0038                                              
                      mlp              5  0.9934 ± 0.0183  0.7370 ± 0.0124  0.8765 ± 0.0052  0.9364 ± 0.0028  0.9086 ± 0.0034  +0.0282  +0.0214  -0.0069  -0.0037   -0.0058
unseen_cell molformer attention        5  1.3090 ± 0.0301  0.9815 ± 0.0217  0.7856 ± 0.0072  0.8868 ± 0.0039  0.8431 ± 0.0043                                              
                      mlp              5  1.3129 ± 0.0349  0.9847 ± 0.0281  0.7843 ± 0.0082  0.8863 ± 0.0051  0.8422 ± 0.0078  +0.0039  +0.0031  -0.0013  -0.0005   -0.0009
            morgan    attention        5  1.3086 ± 0.0314  0.9813 ± 0.0236  0.7857 ± 0.0072  0.8871 ± 0.0039  0.8430 ± 0.0042                                              
                      mlp              5  1.3080 ± 0.0353  0.9804 ± 0.0238  0.7859 ± 0.0082  0.8871 ± 0.0045  0.8437 ± 0.0048  -0.0006  -0.0009  +0.0002  +0.0001   +0.0007
unseen_drug molformer attention        5  2.5363 ± 0.1972  1.9200 ± 0.1498  0.1799 ± 0.0903  0.4954 ± 0.0653  0.4373 ± 0.0495                                              
                      mlp              5  2.5669 ± 0.1811  1.9478 ± 0.0887  0.1568 ± 0.1203  0.4875 ± 0.0970  0.4351 ± 0.0597  +0.0305  +0.0278  -0.0231  -0.0079   -0.0022
            morgan    attention        5  2.6530 ± 0.2463  1.9872 ± 0.1436  0.1024 ± 0.1185  0.4226 ± 0.1014  0.3818 ± 0.0896                                              
                      mlp              5  2.6157 ± 0.2472  1.9787 ± 0.1599  0.1275 ± 0.1171  0.4449 ± 0.1083  0.3996 ± 0.0839  -0.0373  -0.0086  +0.0251  +0.0223   +0.0177

### 2.5 Competitor benchmark (GDSC2 nested CV)
OmicsDRP rows: baseline (morgan) and the final framework (molformer), both attention + all omics.

In [21]:
final = dict(BASE, drug='molformer')
pick = lambda c: (cv[CFG_COLS] == pd.Series(c)).all(axis=1)
bench = pd.concat([cv[~cv.model.str.startswith(('omicsdrp', 'naive'))],
                   cv[pick(BASE)].assign(model='omicsdrp (morgan)'),
                   cv[pick(final)].assign(model='omicsdrp (molformer, final)')])
bs = summarize(bench, ['split_mode', 'model'])
bs = bs.join(bench.groupby(['split_mode', 'model'])['n_params'].first(), on=['split_mode', 'model'])
for sm in SPLITS:
    print(f'\n--- {sm} ---')
    display(pm(bs[bs.split_mode == sm].sort_values('rmse_mean')).drop(columns='split_mode'))


--- mixed ---


,model,n_folds,n_params,rmse,mae,r2,pearson,spearman
6,omicsdrp (morgan),5,7758537,0.9652 ± 0.0235,0.7156 ± 0.0177,0.8834 ± 0.0065,0.9401 ± 0.0035,0.9143 ± 0.0038
5,"omicsdrp (molformer, final)",5,7824073,0.9688 ± 0.0083,0.7171 ± 0.0060,0.8826 ± 0.0029,0.9398 ± 0.0015,0.9139 ± 0.0017
0,deeptta,5,22352193,1.0460 ± 0.0271,0.7789 ± 0.0195,0.8630 ± 0.0080,0.9292 ± 0.0043,0.8990 ± 0.0047
3,graphdrp_gat_gcn,5,4059845,1.1371 ± 0.0580,0.8491 ± 0.0482,0.8379 ± 0.0162,0.9156 ± 0.0088,0.8823 ± 0.0101
4,graphdrp_gcn,5,1373039,1.1661 ± 0.0165,0.8719 ± 0.0131,0.8298 ± 0.0053,0.9113 ± 0.0031,0.8769 ± 0.0033
1,graphdrp,5,873761,1.2253 ± 0.0157,0.9305 ± 0.0128,0.8121 ± 0.0059,0.9077 ± 0.0031,0.8732 ± 0.0030
7,paccmann,5,7965931,1.2693 ± 0.0574,0.9744 ± 0.0566,0.8062 ± 0.0203,0.9124 ± 0.0059,0.8781 ± 0.0071
2,graphdrp_gat,5,1033773,1.4025 ± 0.0988,1.0413 ± 0.0826,0.7527 ± 0.0370,0.8802 ± 0.0143,0.8404 ± 0.0197



--- unseen_cell ---


,model,n_folds,n_params,rmse,mae,r2,pearson,spearman
8,deeptta,5,22352193,1.2714 ± 0.0219,0.9492 ± 0.0179,0.7977 ± 0.0057,0.8938 ± 0.0035,0.8526 ± 0.0040
14,omicsdrp (morgan),5,7758537,1.3086 ± 0.0314,0.9813 ± 0.0236,0.7857 ± 0.0072,0.8871 ± 0.0039,0.8430 ± 0.0042
13,"omicsdrp (molformer, final)",5,7824073,1.3090 ± 0.0301,0.9815 ± 0.0217,0.7856 ± 0.0072,0.8868 ± 0.0039,0.8431 ± 0.0043
9,graphdrp,5,873761,1.5181 ± 0.0254,1.1675 ± 0.0209,0.7116 ± 0.0070,0.8506 ± 0.0055,0.7934 ± 0.0084
11,graphdrp_gat_gcn,5,4059845,1.5332 ± 0.0250,1.1682 ± 0.0220,0.7059 ± 0.0068,0.8420 ± 0.0045,0.7807 ± 0.0062
12,graphdrp_gcn,5,1373039,1.5360 ± 0.0376,1.1751 ± 0.0350,0.7048 ± 0.0108,0.8415 ± 0.0061,0.7792 ± 0.0088
15,paccmann,5,7965931,1.5386 ± 0.1040,1.1787 ± 0.0944,0.7218 ± 0.0313,0.8593 ± 0.0163,0.8065 ± 0.0216
10,graphdrp_gat,5,1033773,1.7144 ± 0.0580,1.3049 ± 0.0476,0.6320 ± 0.0222,0.8121 ± 0.0120,0.7446 ± 0.0156



--- unseen_drug ---


,model,n_folds,n_params,rmse,mae,r2,pearson,spearman
21,"omicsdrp (molformer, final)",5,7824073,2.5363 ± 0.1972,1.9200 ± 0.1498,0.1799 ± 0.0903,0.4954 ± 0.0653,0.4373 ± 0.0495
18,graphdrp_gat,5,1033773,2.6371 ± 0.3127,1.9988 ± 0.2175,0.1119 ± 0.1527,0.3681 ± 0.1814,0.3444 ± 0.1464
22,omicsdrp (morgan),5,7758537,2.6530 ± 0.2463,1.9872 ± 0.1436,0.1024 ± 0.1185,0.4226 ± 0.1014,0.3818 ± 0.0896
19,graphdrp_gat_gcn,5,4059845,2.6911 ± 0.1284,2.0361 ± 0.1134,0.0750 ± 0.0917,0.4135 ± 0.0762,0.3692 ± 0.0533
20,graphdrp_gcn,5,1373039,2.7567 ± 0.1614,2.1167 ± 0.1204,0.0317 ± 0.0771,0.3172 ± 0.1268,0.2539 ± 0.0840
16,deeptta,5,22352193,2.7642 ± 0.2027,2.0955 ± 0.1589,0.0199 ± 0.1581,0.3888 ± 0.1114,0.3472 ± 0.0641
17,graphdrp,5,873761,2.7663 ± 0.1786,2.1245 ± 0.1348,0.0258 ± 0.0704,0.3561 ± 0.0450,0.2769 ± 0.0679
23,paccmann,5,7965931,3.0688 ± 0.8263,2.4831 ± 0.8085,0.0721 ± 0.2266,0.5375 ± 0.0910,0.5017 ± 0.0943


### 2.6 External validation: DepMap (CCLE omics) x PRISM
`ensemble` = 5-fold averaged prediction (the reported number); `fold mean ± std` = each fold model alone.

In [22]:
lab = ext.model.where(ext.tag.isna(), ext.model + ' | ' + ext.tag.fillna(''))
ext = ext.assign(name=lab)
ens = ext[ext.fold == 'ensemble'].set_index(['dataset', 'name'])[METRICS].add_prefix('ens_')
fs = summarize(ext[ext.fold != 'ensemble'], ['dataset', 'name']).set_index(['dataset', 'name'])
for ds in sorted(ext.dataset.unique()):
    t = ens.loc[ds].join(pm(fs.loc[ds].reset_index()).set_index('name')
                         .add_prefix('fold_')).sort_values('ens_pearson', ascending=False)
    print(f'\n--- PRISM-{ds.upper()} (n_pairs={int(ext[ext.dataset == ds].n_pairs.iloc[0])}) ---')
    display(t.round(4))


--- PRISM-HTS (n_pairs=223411) ---


,ens_rmse,ens_mae,ens_r2,ens_pearson,ens_spearman,fold_n_folds,fold_rmse,fold_mae,fold_r2,fold_pearson,fold_spearman
name,,,,,,,,,,,
omicsdrp | SNP+MET+CNV+RNA__attention__molformer__mixed__6548ee,7.8364,3.3818,-0.1802,0.0990,0.2559,5,7.8553 ± 0.0382,3.4103 ± 0.0871,-0.1859 ± 0.0115,0.0955 ± 0.0027,0.2479 ± 0.0061
omicsdrp | MET+CNV+RNA__attention__morgan__mixed__6589e6,7.8636,3.4171,-0.1884,0.0918,0.2415,5,7.8791 ± 0.0261,3.4393 ± 0.0522,-0.1931 ± 0.0079,0.0890 ± 0.0017,0.2329 ± 0.0071
omicsdrp | SNP+MET+CNV+RNA__mlp__morgan__mixed__444b73,7.8042,3.2786,-0.1705,0.0914,0.2412,5,7.8171 ± 0.0255,3.2977 ± 0.0636,-0.1744 ± 0.0077,0.0889 ± 0.0030,0.2351 ± 0.0103
omicsdrp | SNP+RNA__attention__morgan__mixed__075302,7.8667,3.4197,-0.1893,0.0908,0.2392,5,7.8812 ± 0.0125,3.4412 ± 0.0308,-0.1937 ± 0.0038,0.0881 ± 0.0013,0.2321 ± 0.0067
omicsdrp | MET+RNA__attention__morgan__mixed__b45cf1,7.8318,3.3328,-0.1788,0.0907,0.2404,5,7.8477 ± 0.0194,3.3564 ± 0.0453,-0.1836 ± 0.0059,0.0879 ± 0.0020,0.2324 ± 0.0052
omicsdrp | SNP+MET+CNV+RNA__attention__morgan__mixed__c94ea3,7.8585,3.3967,-0.1869,0.0906,0.2372,5,7.8737 ± 0.0224,3.4188 ± 0.0439,-0.1915 ± 0.0068,0.0879 ± 0.0018,0.2286 ± 0.0077
omicsdrp | SNP+MET+RNA__attention__morgan__mixed__021640,7.8398,3.3567,-0.1812,0.0906,0.2401,5,7.8552 ± 0.0419,3.3785 ± 0.0932,-0.1859 ± 0.0126,0.0880 ± 0.0022,0.2329 ± 0.0045
omicsdrp | SNP+CNV+RNA__attention__morgan__mixed__ad3e70,7.8645,3.4051,-0.1887,0.0906,0.2384,5,7.8798 ± 0.0158,3.4274 ± 0.0344,-0.1933 ± 0.0048,0.0878 ± 0.0021,0.2305 ± 0.0048
omicsdrp | CNV+RNA__attention__morgan__mixed__47a229,7.8761,3.4296,-0.1922,0.0903,0.2388,5,7.8911 ± 0.0288,3.4525 ± 0.0680,-0.1967 ± 0.0087,0.0876 ± 0.0017,0.2323 ± 0.0057



--- PRISM-MTS (n_pairs=39433) ---


,ens_rmse,ens_mae,ens_r2,ens_pearson,ens_spearman,fold_n_folds,fold_rmse,fold_mae,fold_r2,fold_pearson,fold_spearman
name,,,,,,,,,,,
omicsdrp | SNP+MET+CNV+RNA__attention__graphormer__mixed__672988,3.5201,2.9191,-1.2323,0.5569,0.4781,5,3.5703 ± 0.0358,2.9530 ± 0.0490,-1.2965 ± 0.0461,0.5371 ± 0.0175,0.4586 ± 0.0250
omicsdrp | SNP+MET+CNV+RNA__attention__molformer__mixed__6548ee,3.6297,3.0032,-1.3734,0.5456,0.4551,5,3.6659 ± 0.0548,3.0252 ± 0.0626,-1.4215 ± 0.0722,0.5313 ± 0.0030,0.4384 ± 0.0101
deeptta,3.4440,2.7876,-1.1368,0.5253,0.4118,5,3.5227 ± 0.0848,2.8384 ± 0.0828,-1.2367 ± 0.1089,0.4957 ± 0.0092,0.3735 ± 0.0360
omicsdrp | MET+RNA__attention__morgan__mixed__b45cf1,3.4895,2.8302,-1.1937,0.5232,0.4326,5,3.5311 ± 0.0478,2.8598 ± 0.0499,-1.2465 ± 0.0608,0.5084 ± 0.0115,0.4173 ± 0.0135
omicsdrp | SNP+MET+CNV+RNA__mlp__morgan__mixed__444b73,3.4983,2.8391,-1.2047,0.5202,0.4286,5,3.5273 ± 0.0355,2.8600 ± 0.0450,-1.2415 ± 0.0451,0.5089 ± 0.0116,0.4173 ± 0.0083
omicsdrp | SNP+RNA__attention__morgan__mixed__075302,3.5485,2.8930,-1.2685,0.5199,0.4339,5,3.5873 ± 0.0188,2.9195 ± 0.0281,-1.3184 ± 0.0243,0.5056 ± 0.0067,0.4184 ± 0.0101
omicsdrp | CNV+RNA__attention__morgan__mixed__47a229,3.5634,2.9039,-1.2876,0.5189,0.4271,5,3.6015 ± 0.0755,2.9300 ± 0.0787,-1.3376 ± 0.0978,0.5052 ± 0.0094,0.4128 ± 0.0124
omicsdrp | MET+CNV+RNA__attention__morgan__mixed__6589e6,3.5933,2.9389,-1.3261,0.5164,0.4185,5,3.6362 ± 0.0451,2.9691 ± 0.0529,-1.3823 ± 0.0595,0.5005 ± 0.0114,0.4013 ± 0.0138
omicsdrp | SNP+MET+RNA__attention__morgan__mixed__021640,3.5177,2.8551,-1.2293,0.5153,0.4249,5,3.5558 ± 0.1015,2.8823 ± 0.0960,-1.2793 ± 0.1301,0.5019 ± 0.0194,0.4110 ± 0.0175
